Installing Libraries: unsloth, transformers e the other dependencies

https://huggingface.co/datasets/nelsntk/mtg-data/viewer/default/rulings

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

Loading the model - I initially used quantized version of Llama but then noticed colab was able to run a not-quantized one. Using a quantized would reduce VRAM but worsen the end results as it reduces the numerical precision of the weights of the model.

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    #"unsloth/Llama-3.2-1B-bnb-4bit",
    #"unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    #"unsloth/Llama-3.2-3B-bnb-4bit",
    #"unsloth/Llama-3.2-3B-Instruct-bnb",
]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = False
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

We will use PEFT (LoRA) to reduce the number of weights during training leveraging the **Unsloth** library.

## Parameter Analysis

### 1. Rank (`r = 16`)

This determines the dimension of the additional matrices.

* **Higher values** (32, 64) help the model learning more complex patterns, but do consume more memory.
* **16** is the standard value balancing performance while saving VRAM.

### 2. Target Modules

Here we select where to inject the LoRA adapters.

* `q_proj`, `k_proj`, `v_proj`: these ones involves the attention mechanism and support the model to understand the context better.
* `gate_proj`, `up_proj`, `down_proj`: These ones affect the Multi-Layer Perceptron.
* Using all these modules (as in this code) returns higher precision during training (which we won't get utilizing only the attention modules).

### 3. LoRA Alpha (`lora_alpha = 16`)

This is a scaling factor for the weights. It determines how much weight to give to the new information compared to the tribal knowledge of the model. Usually is the same as `r` or `r*2`.


### 4. Unsloth specific optimization

* **`lora_dropout = 0`**: Value recommended by Unsloth to maximize speed. Dropout is usually adopted to avoid overfitting, but with LoRA is usually redundant.
* **`use_gradient_checkpointing = "unsloth"`**: This one reduces VRAM usage up to 30% allowing to train larger models or use larger text sequences. Questa è la "magia" di Unsloth. Riduce drasticamente l'uso della memoria VRAM.
* **`bias = "none"`**: It doesn't update neurons' bias, saving memory without losing precision.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.2.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


This code uploads our dataset: https://huggingface.co/datasets/nelsntk/mtg-data. The "mtg-data" dataset is a collection of prompts and responses related to Magic: The Gathering (MTG), a popular collectible card game.

The dataset contains various types of question and answer pairs, including official rulings as responses and corresponding questions generated by GPT-3.5, Q&A data scraped from the web, glossary terms alongside their descriptions, and official rules formatted into Q/A pairs.

This dataset is designed to facilitate the development and research of AI models focused on understanding game dynamics, card interactions, and providing judge rulings.

This code 1) imports the dataset, 2) adds the Instruction column (not available in the original data) and 3) print the first 10 rows together with some information about the data structure.

In [ ]:
import pandas as pd

splits = {'glossary': 'data/glossary-00000-of-00001-7fd5dfc142a99f68.parquet', 'rules': 'data/rules-00000-of-00001-68bab3820a34ffdc.parquet', 'rules_qna': 'data/rules_qna-00000-of-00001-0d40a09c5ba53a11.parquet', 'rulings': 'data/rulings-00000-of-00001-6b771bab5aae4adc.parquet', 'scraped': 'data/scraped-00000-of-00001-2aa5ac90e80d98a1.parquet'}
dataset = pd.read_parquet("hf://datasets/nelsntk/mtg-data/" + splits["glossary"])
dataset['Instruction']='You are a friendly Magic the Gathering level 3 judge. Please provide me a ruling for the following cards.'
print(dataset.head(10))
print(dataset.info())

                                           prompt  \
0                                What is Abandon?   
1                                What is Ability?   
2                           What is Ability Word?   
3                                 What is Absorb?   
4                               What is Activate?   
5                      What is Activated Ability?   
6                        What is Activation Cost?   
7                          What is Active Player?   
8  What is Active Player, Nonactive Player Order?   
9                            What is Active Team?   

                                            response  \
0  To turn a face-up ongoing scheme card face dow...   
1  1. Text on an object that explains what that o...   
2  An italicized word with no rules meaning that ...   
3  A keyword ability that prevents damage. See ru...   
4  To put an activated ability onto the stack and...   
5  A kind of ability. Activated abilities are wri...   
6  Everything that appea

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

dataset = Dataset.from_pandas(dataset)

dataset_split = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

print(f"Dimensions of the Training set: {len(dataset_split['train'])}")
print(f"Dimensions of the Test set: {len(dataset_split['test'])}")

# Saving the 2 files in JSONL format

dataset_split['train'].to_json("train_data.jsonl")
dataset_split['test'].to_json("test_data.jsonl")

Dimensions of the Training set: 523
Dimensions of the Test set: 131


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

41906

This code block is fundamental to the pre-processing phase, as it transforms the raw data of your dataset into a “conversational” format that the model can understand.

Models such as Llama 3.1 do not read plain text; instead, they expect special tags to identify when the user is speaking and when the assistant is responding (e.g. `<|begin_of_text|>`, `<|start_header_id|>`). This Unsloth function configures the tokenizer so that messages are automatically wrapped in the correct Llama 3.1 format.

The format_chat_template(example) function takes the classic dataset columns (instruction, input, output) and organizes them into a list of dictionaries. This is the standard format required by modern libraries such as Hugging Face `trl`.

**System**: Defines the behavior or context (the instruction).

**User**: The user’s question or command.

**Assistant**: The expected response that the model must learn to generate.

## Why is this a critical step?

If we were to skip this step or use the wrong template (for example, using a Mistral template with Llama), the model would struggle significantly during training because it would not recognize the boundaries between questions and answers. By using unsloth, you ensure that the tags are perfectly aligned with the model’s architecture.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def format_chat_template(example):
    # Defining message structure
    messages = [
        {"role": "system", "content": example["Instruction"]},
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["response"]}

    ]

    # Return a new col called 'messages'
    return {"messages": messages}

# Applying the function to the split dataset (train and test)
dataset_formatted = dataset_split.map(
    format_chat_template,
    remove_columns=dataset_split["train"].column_names # Rimuove le vecchie colonne instruction, input, output
)

# Visualizing first record within each dataset
print(dataset_formatted["train"][0])
print(dataset_formatted["test"][0])

Map:   0%|          | 0/523 [00:00<?, ? examples/s]

Map:   0%|          | 0/131 [00:00<?, ? examples/s]

{'messages': [{'content': 'You are a friendly Magic the Gathering level 3 judge. Please provide me a ruling for the following cards.', 'role': 'system'}, {'content': 'What is Phasing?', 'role': 'user'}, {'content': 'A keyword ability that causes a permanent to sometimes be treated as though it does not exist. See rule 702.26, "Phasing."', 'role': 'assistant'}]}
{'messages': [{'content': 'You are a friendly Magic the Gathering level 3 judge. Please provide me a ruling for the following cards.', 'role': 'system'}, {'content': 'What is Evasion Ability?', 'role': 'user'}, {'content': 'An ability that restricts what creatures can block an attacking creature. See rules 509.1b–c.', 'role': 'assistant'}]}


In [ ]:
def apply_tokenizer_template(example):
    # Transforms the list of messages with the final string with tags (es. <|begin_of_text|>)
    full_prompt = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return {"text": full_prompt}

dataset_final = dataset_formatted.map(apply_tokenizer_template)
print(dataset_final["train"][0]["text"])
print(dataset_final["test"][0]["text"])

Map:   0%|          | 0/523 [00:00<?, ? examples/s]

Map:   0%|          | 0/131 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are a friendly Magic the Gathering level 3 judge. Please provide me a ruling for the following cards.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is Phasing?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

A keyword ability that causes a permanent to sometimes be treated as though it does not exist. See rule 702.26, "Phasing."<|eot_id|>
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are a friendly Magic the Gathering level 3 judge. Please provide me a ruling for the following cards.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is Evasion Ability?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

An ability that restricts what creatures can block an attacking creature. See rules 509.1b–c.<|eot_id|>


Let's now configure our training orchestrator `SFTTrainer` (Supervised Fine-Tuning Trainer). This component will put together our model, the data and the mathematical rules to update the weights.

Below its key elements split by function:

### 1. The Trainer

* **`model` e `tokenizer**`: The model and its tokenizer
* **`train_dataset`**: Our training data prepared in the previous steps
* **`dataset_text_field = "text"`**: This indicates our trainer which of the columns contain the text to learn.
* **`DataCollatorForSeq2Seq`**: Manages the dynamic padding, ensuring all the input text in a batch have the same lenght, hence filling the voids and optimizing memory usage.

### 2. Efficency and memory management (`SFTConfig`)

* **`per_device_train_batch_size = 4`**: How many examples will the GPU elaborate.
* **`gradient_accumulation_steps = 2`**: Simulates a larger batch (4 x 2 = 8), accumulating calculations before updating the weights. It is useful to save VRAM.
* **`optim = "adamw_8bit"`**: Utilizes an optimized 8-bit AdamW algorithm, again reducing memory consumption compared to the standard version.

### 3. Learning dynamics

* **`learning_rate`**: Model's speed to learn. High values might break the model, while low ones make it very slow.
* **`max_steps`**: the number of total iterations.
* **`lr_scheduler_type`**: with Linear, the learning speed will go down the more we will reach the end of the training.

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_final["train"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = True, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        warmup_steps = 20,
        num_train_epochs = 4,
        max_steps = -1,
        learning_rate = 5e-5,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/523 [00:00<?, ? examples/s]

During training, we force the trainer to compute the loss only on the generated response and not on the user’s request.
In simple terms, this function tells the model: **“Don’t waste time learning how the user writes, focus only on the response you need to give.”**

Here are the main details:

* **Instruction Masking:** Normally, models compute the error (loss) on every single word of the conversation. With this code, the parts contained in `instruction_part` are ignored during gradient computation.
* **Focus on the Response:** The model is penalized only if it makes mistakes in the words that follow the `response_part` tag. This prevents the model from “memorizing” the dataset questions and pushes it to better understand how to generate the correct response.
* **Efficiency:** By reducing what the model needs to compute, training becomes slightly cleaner and more focused on the desired output.

**What does this mean?**
You are using the specific **Llama-3.1** tags. The trainer will search the text for blocks that start with the `user` header and will “turn off” learning for those segments, reactivating it only when it encounters the `assistant` header.

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=6):   0%|          | 0/523 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/523 [00:00<?, ? examples/s]

In [ ]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\nYou are a friendly Magic the Gathering level 3 judge. Please provide me a ruling for the following cards.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is Attacks and Isn\'t Blocked?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nAn ability that triggers when a creature "attacks and isn\'t blocked" triggers when the creature becomes an unblocked attacking creature. See rule 509.1h.<|eot_id|>'

This code is simply a **debug** tool. It is used to visualize exactly what the model "sees" during training and, most importantly, which parts of the text it will actually be trained on.

Here are the two steps explained simply:

### 1. Space Identification

```python
space = tokenizer(" ", add_special_tokens = False).input_ids[0]

```

This line retrieves the numeric ID that the tokenizer assigns to the "space" character. It is used as a visual "placeholder" in the next step.

### 2. Selective Decoding of Labels

```python
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

```

In detail:

* **The value -100:** In PyTorch and Hugging Face libraries, the value `-100` is a special code that means **"ignore this token when computing the loss"**.
* **The logic:** The code iterates over the labels (`labels`) of the sixth example in the dataset (`[5]`). If a token is `-100` (i.e., it is part of the user instruction that we previously masked), it replaces it with a space. If instead it is a token from the response, it keeps it as is.
* **The result:** When you run `tokenizer.decode`, you will see a string where the user instruction has disappeared (replaced by empty spaces) and **only the assistant’s response is visible**.

### Why is it useful?

It allows you to visually verify that the `train_on_responses_only` function is working correctly. If you see only the assistant’s response, it means the model is not "wasting energy" learning the input, but is focusing only on the output.

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                                                  An ability that triggers when a creature "attacks and isn\'t blocked" triggers when the creature becomes an unblocked attacking creature. See rule 509.1h.<|eot_id|>'

Let's start with training.

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 523 | Num Epochs = 4 | Total steps = 264
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,2.917600
10,2.876300
15,2.696100
20,2.551800
25,2.228100
30,2.135900
35,1.983900
40,1.934400
45,1.891700
50,1.828000


Printing the first 2 messages of the test set we will utilize to verify model's output. These are the system prompt and user request.

In [ ]:
print(dataset_final['test'][0]['messages'][0])
print(dataset_final['test'][0]['messages'][1])

{'content': 'You are a friendly Magic the Gathering level 3 judge. Please provide me a ruling for the following cards.', 'role': 'system'}
{'content': 'What is Evasion Ability?', 'role': 'user'}


This code is used to **test the trained model** (inference phase) to verify how it responds to a question from the test set.

Here is a concise explanation of the steps:

### 1. Optimization and Preparation

* **`FastLanguageModel.for_inference(model)`**: Activates a specific Unsloth optimization that doubles text generation speed, preparing the model to respond instead of learn.
* **`chat_template`**: Ensures that the tokenizer uses the exact Llama-3.1 format to correctly delimit messages.

### 2. Input Preparation

* **`messages`**: Extracts the first two messages from the test dataset (usually the `system` and `user` messages). It does **not** include the correct answer, because we want the model to generate it.
* **`apply_chat_template`**: Converts the list of messages into "tokens" (numbers) understandable by the model.
* `add_generation_prompt = True`: Fundamental — it appends the `<|start_header_id|>assistant<|end_header_id|>\n\n` tag at the end of the sequence to force the model to start responding.

### 3. Response Generation

* **`model.generate`**: The command that actually generates the text.
* `max_new_tokens = 512`: Limits the maximum length of the response.
* `temperature = 1.5`: A high value that makes the model more "creative" and less repetitive.
* `min_p = 0.1`: A sampling technique that discards very unlikely words, improving text coherence.

### 4. Final Decoding

* **`tokenizer.batch_decode(outputs)`**: Converts the numbers (tokens) generated by the model back into readable text (words).

**In summary:** You are asking the model: *"Based on this system and this user question, what would you answer?"* and displaying the result in a readable way.

We can also try adjusting the `temperature` or `min_p` parameters to improve quality of the output.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    dataset_final['test'][0]['messages'][0],
    dataset_final['test'][0]['messages'][1]
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True,
                         temperature = 0.5, min_p = 0.1)
tokenizer.batch_decode(outputs)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\nYou are a friendly Magic the Gathering level 3 judge. Please provide me a ruling for the following cards.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is Evasion Ability?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nAn ability that prevents a creature from being attacked. See rule 702.25, "Evasion."<|eot_id|>']

In [ ]:
print(dataset_final['test'][0]['messages'][2]['content'])

An ability that restricts what creatures can block an attacking creature. See rules 509.1b–c.


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00


**BLEU** (Bilingual Evaluation Understudy) is a metric that measures the similarity between AI-generated text and a human reference text.

It is mainly based on the concept of **n-gram precision** (sequences of n tokens):

1. **Fragment comparison**: It counts how many single words, word pairs, or triplets (n-grams) from the generated text appear identically in the reference text.
2. **Brevity penalty**: If the model generates a correct but overly short answer, the score is reduced to prevent the AI from "cheating" by writing only a few safe words.
3. **Score**: Ranges from **0 to 1** (or from 0 to 100). The higher it is, the more similar the response is to the human one.

**In summary:** BLEU is excellent for understanding whether the model is using the correct vocabulary, but it does not evaluate whether the text is logically coherent or whether the tone is appropriate.

In [ ]:
import evaluate

# 1. Uploading BLEU
bleu = evaluate.load("bleu")

# 2. Preparing lists for predictions and references
predictions = []
references = []

# Inferring on a portion of the test set
for i in range(100):
    # Prepariamo l'input (System + User)
    messages = [
        dataset_final['test'][i]['messages'][0],
        dataset_final['test'][i]['messages'][1]
    ]

    # Computing response
    ground_truth = dataset_final['test'][i]['messages'][2]['content']

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    # Generation
    outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True)

    # Decoding (removing special tokens)
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]

    # Extracting only the response (after the input) - depends on the template
    response_only = generated_text.split("assistant\n\n")[-1]
    print(response_only)

    predictions.append(response_only)
    references.append([ground_truth]) # BLEU needs potential references

# 3. Score calculation
results = bleu.compute(predictions=predictions, references=references)

print(f"BLEU Score: {results['bleu']:.4f}")

An ability that makes a permanent "unblockable." See rule 702.21, "Evasion."
To change a permanent's type from a creature to a noncreature permanent, or vice versa. See rule 305.2.
A variant of the cycling ability that allows a player to pay a cost to turn a card type face up. See rule 702.25, "Cycling."
A permanent is a card in your hand or on the stack that's not a spell. A spell is a card in your hand.
A card type. See rule 312, "Monarch Cards."
A variant in which two teams of players compete against each other. See rule 103, "Variant Rules."
A keyword ability that causes a player who's dealt combat damage by a creature with this ability to discard a card. See rule 702.45, "Poisonous."
A keyword ability that lets a player cast a spell for an additional cost if they control a creature with the same name. See rule 702.26, "Surge."
A keyword ability that lets a creature defend itself in combat. See rule 702.46, "Bushido."
A keyword ability that causes creatures to enter the battlefield

In [ ]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=8f21c2c69a03fabea7af1c73b1f03fa9475259b2614bece04d7519aad08d09e2
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


**ROUGE** metric (Recall-Oriented Understudy for Gisting Evaluation) measures how much of the reference text has been correctly "recovered" by the model.

Unlike BLEU, which rewards precision, ROUGE focuses on **completeness (Recall)**:

1. **Coverage**: It counts how many elements (n-grams) present in the human response have been included in the model’s response.
2. **Main variants**:
* **ROUGE-1**: Overlap of single words.
* **ROUGE-2**: Overlap of word pairs (bigrams).
* **ROUGE-L**: Measures the longest common word sequence between the two texts (Longest Common Subsequence), useful for evaluating sentence structure.


3. **Usage**: It is the standard metric for evaluating **summarization** tasks and long-text generation, because it rewards the model for not forgetting important information.

**In summary:** If BLEU asks “Is what you wrote correct?”, ROUGE asks “Did you write everything that was necessary?”.

In [ ]:
# 1. Uploading the ROUGE metric
rouge = evaluate.load("rouge")

predictions = []
references = []

# 2. Generating responses
for i in range(100):
    messages = [
        dataset_final['test'][i]['messages'][0],
        dataset_final['test'][i]['messages'][1]
    ]
    ground_truth = dataset_final['test'][i]['messages'][2]['content']

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True)

    # Cleaning generated text
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]
    response_only = generated_text.split("assistant\n\n")[-1]

    predictions.append(response_only)
    references.append(ground_truth)

# 3. Score calculation
results = rouge.compute(predictions=predictions, references=references)

# Results visualization
print(f"ROUGE-1: {results['rouge1']:.4f}") # Unigrammi (singole parole)
print(f"ROUGE-2: {results['rouge2']:.4f}") # Bigrammi (coppie di parole)
print(f"ROUGE-L: {results['rougeL']:.4f}") # Sequenza comune più lunga

ROUGE-1: 0.3841
ROUGE-2: 0.1732
ROUGE-L: 0.3449


In [ ]:
!pip install bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.4 MB/s eta 0:00:00


**BERTScore** is a more recent evaluation metric that, unlike BLEU or ROUGE, is not based on exact word matching (n-grams), but on semantic similarity.

It uses BERT models to compare the meaning of word vectors (embeddings). This means that if the model uses a synonym, BERTScore will still yield a high score, while BLEU/ROUGE would yield a zero.

In [ ]:
# 1. Upload the BERTscore metric
bertscore = evaluate.load("bertscore")

predictions = []
references = []

# 2. Generate responses
for i in range(100):
    messages = [
        dataset_final['test'][i]['messages'][0],
        dataset_final['test'][i]['messages'][1]
    ]
    ground_truth = dataset_final['test'][i]['messages'][2]['content']

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True)

    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]
    response_only = generated_text.split("assistant\n\n")[-1]

    predictions.append(response_only)
    references.append(ground_truth)

# 3. Score calculation
results = bertscore.compute(predictions=predictions, references=references, lang="en")

# 4. Visualization of the average results
import numpy as np
print(f"BERTScore Precision: {np.mean(results['precision']):.4f}")
print(f"BERTScore Recall: {np.mean(results['recall']):.4f}")
print(f"BERTScore F1: {np.mean(results['f1']):.4f}")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore Precision: 0.9159
BERTScore Recall: 0.8972
BERTScore F1: 0.9063


In [ ]:
# Testing a prediction

# 1. Ensure correct chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

FastLanguageModel.for_inference(model)  # 2x faster inference

# 2. Define your MTG rules question
messages = [
    {
        "role": "system",
        "content": "You are a Magic: The Gathering judge and rules expert. Answer clearly and precisely."
    },
    {
        "role": "user",
        "content": (
            "How does the commander tax work?"
        )
    }
]

# 3. Apply chat template
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# 4. Generate answer
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.4,
    min_p=0.1,
)

# 5. Decode cleanly
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print(response)

system

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are a Magic: The Gathering judge and rules expert. Answer clearly and precisely.user

How does the commander tax work?assistant

The commander tax is a tax that may be paid when a player casts a commander card. See rule 702.18, "Commander Tax."


In [ ]:
# Save model

save_dir = "mtg_lora_model"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

('mtg_lora_model/tokenizer_config.json',
 'mtg_lora_model/special_tokens_map.json',
 'mtg_lora_model/chat_template.jinja',
 'mtg_lora_model/tokenizer.json')